<a href="https://colab.research.google.com/github/Keerthivasan004/Docker-world/blob/main/spellt5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pyspellchecker language_tool_python transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00


In [3]:
!apt-get install -y default-jre

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data libatspi2.0-0
  libxcomposite1 libxtst6 libxxf86dga1 openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libnss-mdns fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core default-jre default-jre-headless fonts-dejavu-core
  fonts-dejavu-extra gsettings-desktop-schemas libatk-bridge2.0-0
  libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data
  libatspi2.0-0 libxcomposite1 libxtst6 libxxf86dga1 openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
0 upgraded, 19 newly in

In [4]:
!java -version


openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [10]:
import time
from spellchecker import SpellChecker
import language_tool_python
from transformers import T5ForConditionalGeneration, AutoTokenizer

print("Loading models...")
load_start = time.time()

spell = SpellChecker()
tool = language_tool_python.LanguageTool('en-US')

model_name = "Vamsi/T5_Paraphrase_Paws"
tokenizer = AutoTokenizer.from_pretrained(model_name, legacy=False)
model = T5ForConditionalGeneration.from_pretrained(model_name)

load_end = time.time()
print(f"Models loaded in {load_end - load_start:.2f}s\n")

def correct_spelling(text):
    words = text.split()
    corrected_words = []
    for word in words:
        corrected = spell.correction(word)
        corrected_words.append(corrected if corrected else word)
    return " ".join(corrected_words)

def correct_grammar(text):
    matches = tool.check(text)
    return language_tool_python.utils.correct(text, matches)

def paraphrase(text):
    input_text = f"paraphrase: {text} </s>"
    encoding = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=256,
        padding="max_length",
        truncation=True
    )
    outputs = model.generate(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"],
        max_length=256,
        num_beams=5,
        num_return_sequences=1,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def full_pipeline(text):
    total_start = time.time()

    print(f"Original:      {text}")

    t1_start = time.time()
    spelled = correct_spelling(text)
    t1_end = time.time()
    print(f"Spell Fixed:   {spelled}  ({t1_end - t1_start:.3f}s)")

    t2_start = time.time()
    corrected = correct_grammar(spelled)
    t2_end = time.time()
    print(f"Corrected:     {corrected}  ({t2_end - t2_start:.3f}s)")

    t3_start = time.time()
    paraphrased = paraphrase(corrected)
    t3_end = time.time()
    print(f"Paraphrased:   {paraphrased}  ({t3_end - t3_start:.3f}s)")

    total_end = time.time()
    print(f"\n--- Total pipeline time: {total_end - total_start:.3f}s ---")

    return {
        "original": text,
        "spell_fixed": spelled,
        "corrected": corrected,
        "paraphrased": paraphrased,
        "time": {
            "spelling": round(t1_end - t1_start, 3),
            "grammar": round(t2_end - t2_start, 3),
            "paraphrasing": round(t3_end - t3_start, 3),
            "total": round(total_end - total_start, 3)
        }
    }

user_input = "hlo I stdnt Benglur"
result = full_pipeline(user_input)

Loading models...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Models loaded in 11.12s

Original:      hlo I stdnt Benglur
Spell Fixed:   ho I stunt bungler  (1.189s)
Corrected:     Ho I stunt bungler  (15.276s)
Paraphrased:   Ho I stunt bungler  (4.871s)

--- Total pipeline time: 21.342s ---


In [5]:
import time
from spellchecker import SpellChecker
import language_tool_python
from transformers import T5ForConditionalGeneration, AutoTokenizer

print("Loading models...")
load_start = time.time()

spell = SpellChecker()
tool = language_tool_python.LanguageTool('en-US')

model_name = "Vamsi/T5_Paraphrase_Paws"
tokenizer = AutoTokenizer.from_pretrained(model_name, legacy=False)
model = T5ForConditionalGeneration.from_pretrained(model_name)

load_end = time.time()
print(f"Models loaded in {load_end - load_start:.2f}s\n")

def correct_spelling(text):
    words = text.split()
    corrected_words = []
    for word in words:
        corrected = spell.correction(word)
        corrected_words.append(corrected if corrected else word)
    return " ".join(corrected_words)

def correct_grammar(text):
    matches = tool.check(text)
    return language_tool_python.utils.correct(text, matches)

def paraphrase(text):
    input_text = f"paraphrase: {text} </s>"
    encoding = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=256,
        padding="max_length",
        truncation=True
    )
    outputs = model.generate(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"],
        max_length=256,
        num_beams=5,
        num_return_sequences=1,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def full_pipeline(text):
    total_start = time.time()

    print(f"Original:      {text}")

    t1_start = time.time()
    spelled = correct_spelling(text)
    t1_end = time.time()
    print(f"Spell Fixed:   {spelled}  ({t1_end - t1_start:.3f}s)")

    t2_start = time.time()
    corrected = correct_grammar(spelled)
    t2_end = time.time()
    print(f"Corrected:     {corrected}  ({t2_end - t2_start:.3f}s)")

    t3_start = time.time()
    paraphrased = paraphrase(corrected)
    t3_end = time.time()
    print(f"Paraphrased:   {paraphrased}  ({t3_end - t3_start:.3f}s)")

    total_end = time.time()
    print(f"\n--- Total pipeline time: {total_end - total_start:.3f}s ---")

    return {
        "original": text,
        "spell_fixed": spelled,
        "corrected": corrected,
        "paraphrased": paraphrased,
        "time": {
            "spelling": round(t1_end - t1_start, 3),
            "grammar": round(t2_end - t2_start, 3),
            "paraphrasing": round(t3_end - t3_start, 3),
            "total": round(total_end - total_start, 3)
        }
    }

user_input = "hlo I stdnt Benglur"
result = full_pipeline(user_input)

Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Models loaded in 36.67s

Original:      hlo I stdnt Benglur
Spell Fixed:   ho I stunt bungler  (0.775s)
Corrected:     Ho I stunt bungler  (8.884s)
Paraphrased:   Ho I stunt bungler  (2.419s)

--- Total pipeline time: 12.079s ---
